## Module 6: Basic NLP with NLTK
## NLP Analysis on Shakespeare's *Hamlet* from Project Gutenberg

**Student Name:** Dev Panchal
**Course:** EAI6000  

In this notebook, we will perform a complete Natural Language Processing (NLP) pipeline on Shakespeare's *Hamlet*, which is available through NLTK's built-in Gutenberg corpus (originally sourced from Project Gutenberg). The steps include:

1. Loading and exploring the corpus
2. Sentence and word tokenization
3. Lowercasing
4. Stopword removal
5. Stemming and Lemmatization
6. Vectorization (Bag-of-Words)
7. Building a Naive Bayes classifier to classify text by genre

---
## 1. Setup and Imports
First, we install and import all the necessary libraries and download the required NLTK data packages.

In [1]:
# Import core libraries
import nltk
import pandas as pd
import numpy as np
from collections import Counter
from string import punctuation

# Download all required NLTK datasets and models
nltk.download('gutenberg')        # Gutenberg corpus with classic literature texts
nltk.download('punkt')            # Tokenizer models for sentence splitting
nltk.download('punkt_tab')        # Additional tokenizer data
nltk.download('stopwords')        # Common stopwords list for English
nltk.download('wordnet')          # WordNet lexical database for lemmatization
nltk.download('averaged_perceptron_tagger')  # POS tagger model
nltk.download('averaged_perceptron_tagger_eng')  # English POS tagger
nltk.download('brown')            # Brown corpus with genre-labeled text

print('All NLTK packages downloaded successfully!')

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng

All NLTK packages downloaded successfully!


[nltk_data]   Unzipping corpora/brown.zip.


---
## 2. Loading the Corpus
NLTK provides several texts from the Gutenberg Project. Let's explore what's available and then focus on Shakespeare's Hamlet.

In [2]:
# Import the Gutenberg corpus from NLTK
from nltk.corpus import gutenberg

# List all available file IDs in the Gutenberg corpus
# Each file ID represents a different literary work
print('Available texts in NLTK Gutenberg corpus:')
print(gutenberg.fileids())

Available texts in NLTK Gutenberg corpus:
['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


In [3]:
# Load Shakespeare's Hamlet using its file ID
hamlet_words = gutenberg.words('shakespeare-hamlet.txt')  # Returns list of word strings
hamlet_sents = gutenberg.sents('shakespeare-hamlet.txt')  # Returns list of sentence lists
hamlet_raw = gutenberg.raw('shakespeare-hamlet.txt')      # Returns the full raw text as a string

# Display basic statistics about the Hamlet text
print(f'Total number of words in Hamlet: {len(hamlet_words)}')
print(f'Total number of sentences in Hamlet: {len(hamlet_sents)}')
print(f'Total number of characters in Hamlet: {len(hamlet_raw)}')

Total number of words in Hamlet: 37360
Total number of sentences in Hamlet: 3106
Total number of characters in Hamlet: 162881


In [4]:
# Preview the first 500 characters of the raw text to see its format
print('First 500 characters of Hamlet:')
print(hamlet_raw[:500])

First 500 characters of Hamlet:
[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not


In [5]:
# Preview the first 5 sentences as lists of words
# Each sentence is stored as a list of individual word strings
print('First 5 sentences of Hamlet (as word lists):')
for i, sent in enumerate(hamlet_sents[:5]):
    print(f'Sentence {i+1}: {sent}')

First 5 sentences of Hamlet (as word lists):
Sentence 1: ['[', 'The', 'Tragedie', 'of', 'Hamlet', 'by', 'William', 'Shakespeare', '1599', ']']
Sentence 2: ['Actus', 'Primus', '.']
Sentence 3: ['Scoena', 'Prima', '.']
Sentence 4: ['Enter', 'Barnardo', 'and', 'Francisco', 'two', 'Centinels', '.']
Sentence 5: ['Barnardo', '.']


In [6]:
# Preview the first 30 words from the corpus
print('First 30 words of Hamlet:')
print(hamlet_words[:30])

First 30 words of Hamlet:
['[', 'The', 'Tragedie', 'of', 'Hamlet', 'by', 'William', 'Shakespeare', '1599', ']', 'Actus', 'Primus', '.', 'Scoena', 'Prima', '.', 'Enter', 'Barnardo', 'and', 'Francisco', 'two', 'Centinels', '.', 'Barnardo', '.', 'Who', "'", 's', 'there', '?']


---
## 3. Tokenization
Tokenization is the process of breaking text into smaller pieces. **Sentence tokenization** splits text into sentences, and **word tokenization** splits sentences into individual words. We use NLTK's `sent_tokenize()` and `word_tokenize()` functions, which are based on the Punkt algorithm (Kiss and Strunk, 2006).

In [7]:
from nltk import sent_tokenize, word_tokenize

# Take a sample passage from Hamlet for demonstration
# We use raw text and extract a meaningful passage
sample_passage = "To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles."

# Apply sentence tokenization - splits the passage into individual sentences
sentences = sent_tokenize(sample_passage)
print('Sentence Tokenization Result:')
for i, sent in enumerate(sentences):
    print(f'  Sentence {i+1}: {sent}')

Sentence Tokenization Result:
  Sentence 1: To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles.


In [8]:
# Apply word tokenization on each sentence
# This splits each sentence into individual word tokens
print('Word Tokenization Result (per sentence):')
for sent in sentences:
    tokens = word_tokenize(sent)
    print(f'  {tokens}')

Word Tokenization Result (per sentence):
  ['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'Whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'The', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'Or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


In [9]:
# Word tokenize the entire passage at once (treating it as one document)
all_tokens = word_tokenize(sample_passage)
print(f'All tokens from sample passage ({len(all_tokens)} tokens):')
print(all_tokens)

All tokens from sample passage (40 tokens):
['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'Whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'The', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'Or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


---
## 4. Lowercasing
Converting all words to lowercase helps us treat words like "The" and "the" as the same word. We do this **after** tokenization because the tokenizer uses capitalization as a cue for sentence boundaries.

In [10]:
# Tokenize and then lowercase each word
# We apply lowercasing after tokenization to preserve tokenizer accuracy
print('Lowercased tokens from sample passage:')
for sent in sent_tokenize(sample_passage):
    lowered_tokens = [word.lower() for word in word_tokenize(sent)]
    print(f'  {lowered_tokens}')

Lowercased tokens from sample passage:
  ['to', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'the', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


In [11]:
# Lowercase all tokens from the full passage
all_tokens_lower = [word.lower() for word in word_tokenize(sample_passage)]
print(f'All lowercased tokens ({len(all_tokens_lower)} tokens):')
print(all_tokens_lower)

All lowercased tokens (40 tokens):
['to', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'the', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


---
## 5. Stopword Removal
Stopwords are common words like "the", "is", "and" that don't carry much meaning. Removing them helps us focus on the important content words in the text.

In [12]:
from nltk.corpus import stopwords

# Load the English stopwords list from NLTK
stopwords_en = set(stopwords.words('english'))  # Using set for faster lookup

# Display the stopwords list
print(f'Total English stopwords in NLTK: {len(stopwords_en)}')
print(f'Sample stopwords: {list(stopwords_en)[:20]}')

Total English stopwords in NLTK: 198
Sample stopwords: ['being', 'aren', 'be', 'an', 'yours', "she's", 'nor', 're', "they'll", 'up', "should've", 'doing', 'over', 'that', "you're", 'for', 'me', 'been', 'd', 'their']


In [13]:
# Combine NLTK stopwords with punctuation marks for more thorough cleaning
# Python's string.punctuation gives us common punctuation characters
stopwords_en_withpunct = stopwords_en.union(set(punctuation))

print(f'Total stopwords (with punctuation): {len(stopwords_en_withpunct)}')

Total stopwords (with punctuation): 230


In [14]:
# Use a more comprehensive stopwords list from stopwords-json
# This covers additional common words that NLTK might miss
stopwords_json = {"en":["a","a's","able","about","above","according","accordingly","across","actually","after","afterwards","again","against","ain't","all","allow","allows","almost","alone","along","already","also","although","always","am","among","amongst","an","and","another","any","anybody","anyhow","anyone","anything","anyway","anyways","anywhere","apart","appear","appreciate","appropriate","are","aren't","around","as","aside","ask","asking","associated","at","available","away","awfully","b","be","became","because","become","becomes","becoming","been","before","beforehand","behind","being","believe","below","beside","besides","best","better","between","beyond","both","brief","but","by","c","c'mon","c's","came","can","can't","cannot","cant","cause","causes","certain","certainly","changes","clearly","co","com","come","comes","concerning","consequently","consider","considering","contain","containing","contains","corresponding","could","couldn't","course","currently","d","definitely","described","despite","did","didn't","different","do","does","doesn't","doing","don't","done","down","downwards","during","e","each","edu","eg","eight","either","else","elsewhere","enough","entirely","especially","et","etc","even","ever","every","everybody","everyone","everything","everywhere","ex","exactly","example","except","f","far","few","fifth","first","five","followed","following","follows","for","former","formerly","forth","four","from","further","furthermore","g","get","gets","getting","given","gives","go","goes","going","gone","got","gotten","greetings","h","had","hadn't","happens","hardly","has","hasn't","have","haven't","having","he","he's","hello","help","hence","her","here","here's","hereafter","hereby","herein","hereupon","hers","herself","hi","him","himself","his","hither","hopefully","how","howbeit","however","i","i'd","i'll","i'm","i've","ie","if","ignored","immediate","in","inasmuch","inc","indeed","indicate","indicated","indicates","inner","insofar","instead","into","inward","is","isn't","it","it'd","it'll","it's","its","itself","j","just","k","keep","keeps","kept","know","known","knows","l","last","lately","later","latter","latterly","least","less","lest","let","let's","like","liked","likely","little","look","looking","looks","ltd","m","mainly","many","may","maybe","me","mean","meanwhile","merely","might","more","moreover","most","mostly","much","must","my","myself","n","name","namely","nd","near","nearly","necessary","need","needs","neither","never","nevertheless","new","next","nine","no","nobody","non","none","noone","nor","normally","not","nothing","novel","now","nowhere","o","obviously","of","off","often","oh","ok","okay","old","on","once","one","ones","only","onto","or","other","others","otherwise","ought","our","ours","ourselves","out","outside","over","overall","own","p","particular","particularly","per","perhaps","placed","please","plus","possible","presumably","probably","provides","q","que","quite","qv","r","rather","rd","re","really","reasonably","regarding","regardless","regards","relatively","respectively","right","s","said","same","saw","say","saying","says","second","secondly","see","seeing","seem","seemed","seeming","seems","seen","self","selves","sensible","sent","serious","seriously","seven","several","shall","she","should","shouldn't","since","six","so","some","somebody","somehow","someone","something","sometime","sometimes","somewhat","somewhere","soon","sorry","specified","specify","specifying","still","sub","such","sup","sure","t","t's","take","taken","tell","tends","th","than","thank","thanks","thanx","that","that's","thats","the","their","theirs","them","themselves","then","thence","there","there's","thereafter","thereby","therefore","therein","theres","thereupon","these","they","they'd","they'll","they're","they've","think","third","this","thorough","thoroughly","those","though","three","through","throughout","thru","thus","to","together","too","took","toward","towards","tried","tries","truly","try","trying","twice","two","u","un","under","unfortunately","unless","unlikely","until","unto","up","upon","us","use","used","useful","uses","using","usually","uucp","v","value","various","very","via","viz","vs","w","want","wants","was","wasn't","way","we","we'd","we'll","we're","we've","welcome","well","went","were","weren't","what","what's","whatever","when","whence","whenever","where","where's","whereafter","whereas","whereby","wherein","whereupon","wherever","whether","which","while","whither","who","who's","whoever","whole","whom","whose","why","will","willing","wish","with","within","without","won't","wonder","would","wouldn't","x","y","yes","yet","you","you'd","you'll","you're","you've","your","yours","yourself","yourselves","z","zero"]}

# Create the combined stopword list using all three sources
stopwords_json_en = set(stopwords_json['en'])
stopwords_nltk_en = set(stopwords.words('english'))
stopwords_punct = set(punctuation)

# Union all stopword sets together for maximum coverage
stoplist_combined = set.union(stopwords_json_en, stopwords_nltk_en, stopwords_punct)

print(f'Combined stopwords list size: {len(stoplist_combined)}')

Combined stopwords list size: 635


In [15]:
# Demonstrate stopword removal on our sample passage
# First tokenize and lowercase, then filter out stopwords and punctuation
tokens_lower = [word.lower() for word in word_tokenize(sample_passage)]

print('Before stopword removal:')
print(tokens_lower)
print(f'Token count: {len(tokens_lower)}')
print()

# Remove stopwords using list comprehension - keep only meaningful words
tokens_clean = [word for word in tokens_lower if word not in stoplist_combined]

print('After stopword removal:')
print(tokens_clean)
print(f'Token count: {len(tokens_clean)}')

Before stopword removal:
['to', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'the', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']
Token count: 40

After stopword removal:
['question', "'t", 'nobler', 'mind', 'suffer', 'slings', 'arrows', 'outrageous', 'fortune', 'arms', 'sea', 'troubles']
Token count: 12


---
## 6. Stemming and Lemmatization
Both stemming and lemmatization try to reduce words to their root form. **Stemming** uses simple rules to chop off word endings, while **lemmatization** uses linguistic knowledge to find the proper dictionary root (lemma) of a word.

In [16]:
from nltk.stem import PorterStemmer

# Initialize the Porter Stemmer (Porter, 1980)
# This is a rule-based stemmer that strips common suffixes
porter = PorterStemmer()

# Demonstrate stemming on a few example words
test_words = ['walking', 'walks', 'walked', 'troubles', 'suffering', 'nobler']
print('Porter Stemmer Results:')
for word in test_words:
    print(f'  {word} -> {porter.stem(word)}')

Porter Stemmer Results:
  walking -> walk
  walks -> walk
  walked -> walk
  troubles -> troubl
  suffering -> suffer
  nobler -> nobler


In [17]:
from nltk.stem import WordNetLemmatizer

# Initialize the WordNet Lemmatizer
# This uses the WordNet lexical database to find root words
wnl = WordNetLemmatizer()

# Without POS tags, the lemmatizer assumes all words are nouns (default)
# This leads to incomplete lemmatization for verbs and adjectives
print('WordNet Lemmatizer Results (without POS tags - default noun):')
for word in test_words:
    print(f'  {word} -> {wnl.lemmatize(word)}')

WordNet Lemmatizer Results (without POS tags - default noun):
  walking -> walking
  walks -> walk
  walked -> walked
  troubles -> trouble
  suffering -> suffering
  nobler -> nobler


In [18]:
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

wnl = WordNetLemmatizer()

def penn2morphy(penntag):
    """
    Converts Penn Treebank POS tags to WordNet POS tags.
    This mapping is needed because NLTK's pos_tag uses Penn tags
    but the WordNet lemmatizer needs WordNet-format tags.
    
    Args:
        penntag (str): A Penn Treebank POS tag (e.g., 'NN', 'VB', 'JJ')
    Returns:
        str: A WordNet POS tag ('n' for noun, 'v' for verb, 'a' for adjective, 'r' for adverb)
    """
    morphy_tag = {'NN': 'n', 'JJ': 'a', 'VB': 'v', 'RB': 'r'}
    try:
        return morphy_tag[penntag[:2]]
    except:
        return 'n'  # Default to noun if no mapping found


def lemmatize_sent(text):
    """
    Lemmatizes a sentence by first POS tagging each word,
    converting the tag to WordNet format, then applying lemmatization.
    
    Args:
        text (str): Input sentence as a string
    Returns:
        list: List of lowercased lemmatized word strings
    """
    return [wnl.lemmatize(word.lower(), pos=penn2morphy(tag))
            for word, tag in pos_tag(word_tokenize(text))]


# Test the lemmatize_sent function on a sample sentence
test_sentence = 'He is walking to school and suffering from troubles'
print(f'Original: {test_sentence}')
print(f'Lemmatized: {lemmatize_sent(test_sentence)}')

Original: He is walking to school and suffering from troubles
Lemmatized: ['he', 'be', 'walk', 'to', 'school', 'and', 'suffer', 'from', 'trouble']


In [19]:
# Apply lemmatization and stopword removal to our Hamlet sample passage
print('Original passage:')
print(sample_passage, '\n')

# Lemmatize and then remove stopwords and digits
hamlet_lemmatized_clean = [word for word in lemmatize_sent(sample_passage)
                           if word not in stoplist_combined
                           and not word.isdigit()]

print('Lemmatized and cleaned:')
print(hamlet_lemmatized_clean)

Original passage:
To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles. 

Lemmatized and cleaned:
['question', "'t", 'nobler', 'mind', 'suffer', 'sling', 'arrow', 'outrageous', 'fortune', 'arm', 'sea', 'trouble']


In [20]:
def preprocess_text(text):
    """
    Complete text preprocessing pipeline that combines:
    1. Word tokenization
    2. Lowercasing
    3. POS tagging
    4. Lemmatization
    5. Stopword and punctuation removal
    6. Digit removal
    
    Args:
        text (str): Raw input text (a document or sentence)
    Returns:
        list: List of cleaned, lemmatized word strings
    """
    return [word for word in lemmatize_sent(text)
            if word not in stoplist_combined
            and not word.isdigit()]

# Test the full preprocessing pipeline
print('Full preprocessing pipeline result:')
print(preprocess_text(sample_passage))

Full preprocessing pipeline result:
['question', "'t", 'nobler', 'mind', 'suffer', 'sling', 'arrow', 'outrageous', 'fortune', 'arm', 'sea', 'trouble']


---
## 7. Applying NLP Pipeline to Full Hamlet Text
Let's apply our preprocessing pipeline to the full Hamlet text and analyze word frequencies.

In [21]:
# Process the full Hamlet text
# We join all words back into a string, then apply our preprocessing function
# Using first 10000 characters to keep runtime reasonable
hamlet_sample_text = hamlet_raw[:10000]

# Apply full preprocessing pipeline to the Hamlet sample
hamlet_processed = preprocess_text(hamlet_sample_text)

print(f'Processed tokens from Hamlet sample: {len(hamlet_processed)}')
print(f'First 30 processed tokens: {hamlet_processed[:30]}')

Processed tokens from Hamlet sample: 860
First 30 processed tokens: ['tragedie', 'hamlet', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'centinels', 'barnardo', "'s", 'fran', 'nay', 'answer', 'stand', 'vnfold', 'selfe', 'bar', 'long', 'liue', 'king', 'fran', 'barnardo', 'bar', 'fran', 'carefully', 'vpon']


In [22]:
# Count word frequencies using Python's Counter class
# Counter creates a dictionary with words as keys and their counts as values
hamlet_word_counts = Counter(hamlet_processed)

# Display the 20 most common meaningful words in Hamlet
print('Top 20 most frequent words in Hamlet (after preprocessing):')
for word, count in hamlet_word_counts.most_common(20):
    print(f'  {word}: {count}')

Top 20 most frequent words in Hamlet (after preprocessing):
  mar: 17
  hor: 14
  'd: 14
  speake: 14
  haue: 13
  king: 12
  barn: 10
  night: 10
  horatio: 9
  's: 8
  fran: 8
  't: 8
  hath: 8
  thou: 8
  bar: 7
  againe: 7
  hamlet: 6
  enter: 6
  barnardo: 6
  watch: 6


---
## 8. From Strings to Vectors (Bag-of-Words)
In the Vector Space Model, we represent each document as a vector of numbers. Each number represents how many times a word appears in that document. This is called the **Bag-of-Words (BoW)** approach.

In [23]:
# Demonstrate Bag-of-Words on two simple sentences
sent1 = "To be, or not to be, that is the question."
sent2 = "Whether tis nobler in the mind to suffer."

# Preprocess both sentences
processed_sent1 = preprocess_text(sent1)
processed_sent2 = preprocess_text(sent2)

print('Processed Sentence 1:', processed_sent1)
print('Word Counts:', Counter(processed_sent1))
print()
print('Processed Sentence 2:', processed_sent2)
print('Word Counts:', Counter(processed_sent2))

Processed Sentence 1: ['question']
Word Counts: Counter({'question': 1})

Processed Sentence 2: ['ti', 'nobler', 'mind', 'suffer']
Word Counts: Counter({'ti': 1, 'nobler': 1, 'mind': 1, 'suffer': 1})


---
## 9. Vectorization with sklearn's CountVectorizer
scikit-learn provides `CountVectorizer` which builds the vocabulary and converts text into count vectors automatically. We can override its tokenizer with our custom `preprocess_text` function.

In [24]:
from io import StringIO
from sklearn.feature_extraction.text import CountVectorizer
from operator import itemgetter

# Two sample sentences for vectorization demonstration
sent1 = "To be, or not to be, that is the question."
sent2 = "Whether tis nobler in the mind to suffer."

# Create a CountVectorizer that uses our custom preprocess_text function
# The analyzer parameter overrides the entire text processing pipeline
with StringIO('\n'.join([sent1, sent2])) as fin:
    count_vect = CountVectorizer(analyzer=preprocess_text)
    count_vect.fit_transform(fin)

# Display the vocabulary that the vectorizer learned
# Each word is mapped to a unique integer index
print('Vocabulary (word -> index):')
print(count_vect.vocabulary_)

Vocabulary (word -> index):
{'question': 2, 'ti': 4, 'nobler': 1, 'mind': 0, 'suffer': 3}


In [25]:
# Transform the sentences into count vectors and display the matrix
# Each row is a sentence, each column is a word from the vocabulary
words_sorted_by_index, _ = zip(*sorted(count_vect.vocabulary_.items(), key=itemgetter(1)))

print('Preprocessed Sentence 1:', preprocess_text(sent1))
print('Preprocessed Sentence 2:', preprocess_text(sent2))
print()
print('Vocabulary (column headers):', words_sorted_by_index)
print()
print('Count Vectors (rows = sentences, columns = word counts):')
print(count_vect.transform([sent1, sent2]).toarray())

Preprocessed Sentence 1: ['question']
Preprocessed Sentence 2: ['ti', 'nobler', 'mind', 'suffer']

Vocabulary (column headers): ('mind', 'nobler', 'question', 'suffer', 'ti')

Count Vectors (rows = sentences, columns = word counts):
[[0 0 1 0 0]
 [1 1 0 1 1]]


---
## 10. Naive Bayes Classification on Brown Corpus
Now we apply everything we learned to build a **Naive Bayes classifier**. Instead of the pizza dataset from the tutorial, we will use the **Brown Corpus** which contains text categorized by genre. Our task is to classify text as either **"news"** or **"fiction"** (a genre classification problem).

In [26]:
from nltk.corpus import brown

# The Brown Corpus has text from 500 sources organized by genre
# Let's see all available genres/categories
print('Available genres in Brown Corpus:')
print(brown.categories())

Available genres in Brown Corpus:
['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']


In [27]:
# We will classify text into two genres: 'news' and 'fiction'
# Extract sentences from each genre and join them into documents

# Get file IDs for each genre
news_fileids = brown.fileids(categories='news')
fiction_fileids = brown.fileids(categories='fiction')

print(f'Number of news files: {len(news_fileids)}')
print(f'Number of fiction files: {len(fiction_fileids)}')

Number of news files: 44
Number of fiction files: 29


In [28]:
# Build a dataset where each document is the raw text of one file
# and the label is the genre (news=1, fiction=0)
documents = []  # Will hold the text of each document
labels = []     # Will hold the genre label for each document

# Add news documents with label 'news'
for fid in news_fileids:
    # Join all words in the file into a single string
    text = ' '.join(brown.words(fileids=fid))
    documents.append(text)
    labels.append('news')

# Add fiction documents with label 'fiction'
for fid in fiction_fileids:
    text = ' '.join(brown.words(fileids=fid))
    documents.append(text)
    labels.append('fiction')

# Create a pandas DataFrame to organize our data
df_brown = pd.DataFrame({'text': documents, 'genre': labels})

print(f'Total documents: {len(df_brown)}')
print(f'\nGenre distribution:')
print(df_brown['genre'].value_counts())
print(f'\nSample document (first 200 chars):')
print(df_brown['text'].iloc[0][:200])

Total documents: 73

Genre distribution:
genre
news       44
fiction    29
Name: count, dtype: int64

Sample document (first 200 chars):
The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place . The jury further said in term-end present


In [29]:
from sklearn.model_selection import train_test_split

# Split the data into training (80%) and validation (20%) sets
# This hold-out validation lets us test the model on unseen data
# random_state=42 ensures reproducible results
train_data, valid_data = train_test_split(df_brown, test_size=0.2, random_state=42)

print(f'Training set size: {len(train_data)}')
print(f'Validation set size: {len(valid_data)}')

Training set size: 58
Validation set size: 15


In [30]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer with our custom preprocessing pipeline
# This will tokenize, lowercase, lemmatize, and remove stopwords
count_vect = CountVectorizer(analyzer=preprocess_text)

# fit_transform on training data: builds vocabulary AND creates count vectors
train_vectors = count_vect.fit_transform(train_data['text'])
train_labels = train_data['genre']

# transform on validation data: uses the SAME vocabulary to create count vectors
# We must NOT fit on validation data to avoid data leakage
valid_vectors = count_vect.transform(valid_data['text'])
valid_labels = valid_data['genre']

print(f'Training vector shape: {train_vectors.shape}')
print(f'  (rows = documents, columns = unique words in vocabulary)')
print(f'Validation vector shape: {valid_vectors.shape}')
print(f'Vocabulary size: {len(count_vect.vocabulary_)}')

Training vector shape: (58, 11371)
  (rows = documents, columns = unique words in vocabulary)
Validation vector shape: (15, 11371)
Vocabulary size: 11371


In [31]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the Multinomial Naive Bayes classifier
# MultinomialNB works well with word count features
clf = MultinomialNB()
clf.fit(train_vectors, train_labels)

print('Multinomial Naive Bayes classifier trained successfully!')

Multinomial Naive Bayes classifier trained successfully!


In [32]:
# Make predictions on the validation set to evaluate model performance
predictions = clf.predict(valid_vectors)

# Calculate and display accuracy
accuracy = accuracy_score(valid_labels, predictions) * 100
print(f'Genre Classification Accuracy: {accuracy:.2f}%')
print()

# Display detailed classification report with precision, recall, and F1-score
# Precision: Of all items predicted as a class, how many were correct?
# Recall: Of all actual items in a class, how many did we find?
# F1-score: Harmonic mean of precision and recall (balanced measure)
print('Detailed Classification Report:')
print(classification_report(valid_labels, predictions))

Genre Classification Accuracy: 100.00%

Detailed Classification Report:
              precision    recall  f1-score   support

     fiction       1.00      1.00      1.00         5
        news       1.00      1.00      1.00        10

    accuracy                           1.00        15
   macro avg       1.00      1.00      1.00        15
weighted avg       1.00      1.00      1.00        15



In [33]:
# Now retrain on the FULL dataset for the best possible model
# More training data generally leads to better model performance

# Reinitialize and fit on all data
count_vect_full = CountVectorizer(analyzer=preprocess_text)
full_vectors = count_vect_full.fit_transform(df_brown['text'])
full_labels = df_brown['genre']

# Train final classifier on all available data
clf_final = MultinomialNB()
clf_final.fit(full_vectors, full_labels)

print(f'Final model trained on {len(df_brown)} documents')
print(f'Final vocabulary size: {len(count_vect_full.vocabulary_)}')

Final model trained on 73 documents
Final vocabulary size: 13177


In [34]:
# Test the final classifier on new custom sentences
# These sentences were not part of the training data
test_sentences = [
    "The president announced new economic policies at the press conference today.",
    "She walked through the dark forest, her heart pounding with every step.",
    "The stock market rallied after the Federal Reserve cut interest rates.",
    "He gazed into her eyes and whispered words she had longed to hear."
]

# Vectorize the test sentences using the same vocabulary
test_vectors = count_vect_full.transform(test_sentences)

# Predict the genre of each test sentence
test_predictions = clf_final.predict(test_vectors)

print('Predictions on New Sentences:')
print('=' * 70)
for sentence, prediction in zip(test_sentences, test_predictions):
    print(f'Text: "{sentence}"')
    print(f'Predicted Genre: {prediction}')
    print('-' * 70)

Predictions on New Sentences:
Text: "The president announced new economic policies at the press conference today."
Predicted Genre: news
----------------------------------------------------------------------
Text: "She walked through the dark forest, her heart pounding with every step."
Predicted Genre: fiction
----------------------------------------------------------------------
Text: "The stock market rallied after the Federal Reserve cut interest rates."
Predicted Genre: news
----------------------------------------------------------------------
Text: "He gazed into her eyes and whispered words she had longed to hear."
Predicted Genre: fiction
----------------------------------------------------------------------


---
## Conclusion

In this notebook, we performed a complete NLP pipeline on text from the Gutenberg Project (Shakespeare's Hamlet) and the Brown Corpus:

1. **Corpus Loading**: We loaded and explored Hamlet from NLTK's Gutenberg corpus.
2. **Tokenization**: We split text into sentences and words using NLTK's Punkt-based tokenizers.
3. **Lowercasing**: We normalized text by converting all tokens to lowercase.
4. **Stopword Removal**: We removed non-content words using combined stopword lists from NLTK and stopwords-json.
5. **Stemming & Lemmatization**: We compared the Porter Stemmer with the WordNet Lemmatizer, noting that lemmatization with POS tags gives more accurate root words.
6. **Vectorization**: We converted text into numerical vectors using Bag-of-Words with sklearn's CountVectorizer.
7. **Classification**: We built a Multinomial Naive Bayes classifier to distinguish news articles from fiction in the Brown Corpus, achieving strong accuracy.

These foundational NLP techniques form the basis for more advanced text analysis and natural language understanding tasks.